# figure.ipynb —— 全部作图代码第一部分（§2 + §3）所有进论文/报告的图，一个 notebook 出全。取代原先散在四处的 `make_figures.py`、`fig1_sales_share.py`、`fig2_lift_plot.py`、`similarity_figures.py`。## 产出清单| 节 | 图 | 用在 ||---|---|---|| §2.1 | `firm_classification.png` + `summary_table.csv/.md` | 报告 §2.1 || §2.2 | `fig2_composition_by_sales_scope_{2017,2018,pooled}.png` | **论文** + 报告 §2.2 || §2.2 | `fig3_gap_by_size_decile_{2017,2018,pooled}.png` | **论文** + 报告 §2.2 || §3.1 | `fig1_combined_shares_vs_{input,output}_similarity.png` | **论文** + 报告 §3.1 || §3.1 | `fig1{A,B,C}_share_*_vs_*_similarity.png`（6 张分图）| 备用 || §3.1 | `fig2_lift_{input,output}_similarity.png`、`fig2_lift_combined.png` | **论文 + deck** || §3.2 | `sim_vs_rank_{all,n_le5,n_6to10,n_11to15}.png`（+combined）| **论文 + deck** || §IO | `fig1_input_vectors.png`、`fig3_vector_angles.png` | **论文 + deck**（★ 重写）|## ★ 关于最后两张原脚本 `Data/seminar_viz.py` **已从机器上删除**，全盘搜不到，也没有 git 历史可恢复。这里是重写版，同时修掉三个问题：1. **换例子** —— 原图用乘用车/电动汽车/碾磨脱壳谷物。老师明确不喜欢；而且电动汽车对乘用车的 demand complementarity 排第 7/2777（p99.59），是**高 S 高 C**，用来讲两个维度的分离是讲反的。2. **改标签** —— `Output Similarity` → `Demand Complementarity`（老师定的术语）。3. **用对系数列** —— 去向向量必须用 `coefficient_cal`。用原始 `coefficient` 会画出"电动汽车 88% 卖给除霜器"这种，因为 raw 表的行和极差 3.27 万倍、且与产品规模**负相关**。## 运行单元格自上而下跑。每节独立，可只跑需要的那节——`§0` 的 loader 有磁盘缓存，重跑很快。

## §0　路径与前置检查切换 VM / 本地只改 `ON_VM`。

In [ ]:
import os, sys, time, gc, warningsfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use("Agg")                      # notebook 里也不弹窗，直接存盘import matplotlib.pyplot as pltimport matplotlib.ticker as mtickerwarnings.filterwarnings("ignore", category=FutureWarning)try:    sys.stdout.reconfigure(encoding="utf-8")except Exception:    pass# ════════════════ 路径 ════════════════ON_VM = Trueif ON_VM:    DATA = Path(r"G:\Kuangyu_Temp\Data")                        # 原始只读    OUT  = Path(r"G:\Kuangyu_Temp\Outsource\Empirical1_data")   # 产物    CODE = Path(r"G:\Kuangyu_Temp\Outsource\Empirical1")        # 代码（git）    SRC  = Path(r"G:\Kuangyu_Temp\Outsource")                   # io_table_lite 等else:    DATA = Path(r"C:\Users\HKUBS\Documents\aproject\Data")    OUT  = Path(r"C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1_data")    CODE = Path(r"C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1")    SRC  = DATAFULL_DATA = OUT  / "full_data.dta"                 # 本 pipeline 产出的新版SIM       = DATA / "full_product_similarity.dta"   # 385.7 万对 S/CIO_CAL    = DATA / "io_table_calibrated.dta"       # ★ 必须是 calibrated 版BIANMA    = DATA / "bianma.dta"                    # 2,778 个 9 位码 + 中文名FIG   = CODE / "results" / "figures"               # 图 → 进 gitCACHE = OUT  / "_figure_cache"                     # 中间件 → 不进 gitFIG.mkdir(parents=True, exist_ok=True)CACHE.mkdir(parents=True, exist_ok=True)# ════════════════ 前置检查 ════════════════NEEDED = {    "full_data.dta":              (FULL_DATA, "§2.1 §2.2 §3.1 §3.2"),    "full_product_similarity.dta":(SIM,       "§3.1 §3.2"),    "io_table_calibrated.dta":    (IO_CAL,    "§IO 两张向量图"),    "bianma.dta":                 (BIANMA,    "§IO 产品中文名"),}print("=" * 74)print(f"{'输入文件':<32}{'状态':<8}{'大小':>10}   用于")print("=" * 74)AVAIL = {}for name, (p, use) in NEEDED.items():    ok = p.exists()    AVAIL[name] = ok    size = f"{p.stat().st_size/1e9:.2f} GB" if ok else "—"    print(f"{name:<32}{'✅ 有' if ok else '❌ 缺':<8}{size:>10}   {use}")print("=" * 74)print(f"图输出 → {FIG}")print(f"缓存   → {CACHE}")if not AVAIL["io_table_calibrated.dta"]:    print("\n⚠️  io_table_calibrated.dta 不在 VM 上 —— §IO 那两张图会跳过。")    print("    从本地 C:\\Users\\HKUBS\\Documents\\aproject\\Data\\ 拷 1.8 GB 过来即可。")    print("    注意：不能用 io_table_lite.dta 替代，它只有 coefficient 一列，没有 coefficient_cal。")

## §0b　共用 loader`full_data.dta` 19 GB，全部分块读。三个 loader 各自带磁盘缓存，第二次跑秒开。**口径说明**：直接用 `full_data` 里存好的 `is_main`（02 已按 `production_value` 最大 + `product_id` 并列打破算出）。旧脚本各自用 `rank(method='first')` 重算，并列时结果不确定——这里统一，不再重算。

In [ ]:
CHUNK = 2_000_000def _cached(name, build):    """带磁盘缓存的 loader。缓存比 full_data 旧就重建。"""    f = CACHE / name    if f.exists() and f.stat().st_mtime >= FULL_DATA.stat().st_mtime:        print(f"  [缓存命中] {name}")        return pd.read_parquet(f)    t0 = time.time()    df = build()    df.to_parquet(f, index=False)    print(f"  [已缓存] {name}   {len(df):,} 行   {time.time()-t0:.0f}s")    return dfdef load_firm_year():    """§2.1 用：firm × year 面板，一行一个企业年。"""    def build():        cols = ["year", "firm_id", "is_intermediary",                "outsourcing_intensity", "firm_total_output"]        parts = []        for i, ch in enumerate(pd.read_stata(FULL_DATA, columns=cols, chunksize=CHUNK), 1):            parts.append(ch.drop_duplicates(subset=["year", "firm_id"]))            if i % 10 == 0: print(f"    chunk {i}")        fy = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["year", "firm_id"])        fy["year"] = fy["year"].astype(int)        return fy.reset_index(drop=True)    return _cached("firm_year.parquet", build)def load_scope_gap():    """§2.2 用：非中介企业的 firm × product × year。"""    def build():        cols = ["firm_id", "year", "product_id", "outsourcing_percen",                "total_output", "firm_total_output", "is_intermediary"]        parts = []        for i, ch in enumerate(pd.read_stata(FULL_DATA, columns=cols, chunksize=CHUNK), 1):            ch = ch[(ch["is_intermediary"] == 0) & (ch["year"].isin([2017, 2018]))]            parts.append(ch.drop(columns=["is_intermediary"]))            if i % 10 == 0: print(f"    chunk {i}  累计 {sum(len(p) for p in parts):,}")        return pd.concat(parts, ignore_index=True)    df = _cached("scope_gap.parquet", build)    EPS = 1e-6    df["category"] = np.where(df["outsourcing_percen"] <= EPS, "pure_self",                       np.where(df["outsourcing_percen"] >= 1 - EPS, "pure_os", "mixed"))    return dfdef load_secondary():    """§3.1 §3.2 共用：非中介企业的副产品行 + 核心产品 id。"""    def build():        cols = ["firm_id", "year", "product_id", "is_intermediary",                "is_main", "main_product", "total_output", "firm_total_output"]        parts = []        for i, ch in enumerate(pd.read_stata(FULL_DATA, columns=cols, chunksize=CHUNK), 1):            ch = ch[(ch["is_intermediary"] == 0) & (ch["is_main"] == 0)]            parts.append(ch.drop(columns=["is_intermediary", "is_main"]))            if i % 10 == 0: print(f"    chunk {i}  累计 {sum(len(p) for p in parts):,}")        d = pd.concat(parts, ignore_index=True)        d = d.rename(columns={"main_product": "main_pid"})        for c in ("main_pid", "product_id"):            d[c] = d[c].astype(str).str.strip()        return d    return _cached("secondary.parquet", build)def load_core():    """核心产品行：firm × year → main_pid + 核心产品销售额。    §3.1 的分母 C 和 §3.1b 的 exposure 共用这一份，全表只扫一遍。"""    def build():        cols = ["firm_id", "year", "product_id", "is_intermediary", "is_main", "total_output"]        parts = []        for i, ch in enumerate(pd.read_stata(FULL_DATA, columns=cols, chunksize=CHUNK), 1):            ch = ch[(ch["is_intermediary"] == 0) & (ch["is_main"] == 1)]            parts.append(ch[["firm_id", "year", "product_id", "total_output"]])            if i % 10 == 0: print(f"    chunk {i}")        d = (pd.concat(parts, ignore_index=True)               .rename(columns={"product_id": "main_pid", "total_output": "main_sales"}))        d["main_pid"] = d["main_pid"].astype(str).str.strip()        return d    return _cached("core.parquet", build)def load_sim_bi():    """相似度表翻成双向。原表每对只存一次。"""    sim = pd.read_stata(SIM, columns=["product_1", "product_2",                                      "input_similarity", "output_similarity"])    rev = sim.rename(columns={"product_1": "product_2", "product_2": "product_1"})    bi = pd.concat([sim, rev], ignore_index=True)    bi = bi.drop_duplicates(subset=["product_1", "product_2"])    bi = bi.rename(columns={"product_1": "main_pid", "product_2": "product_id"})    for c in ("main_pid", "product_id"):        bi[c] = bi[c].astype(str).str.strip()    print(f"  双向 similarity: {len(bi):,} 对")    return biprint("loader 就绪")

## §2.1　企业分类三分类（firm-year 层面）：- **Intermediary** —— `is_intermediary == 1`，即外包强度 > 0.90- **Outsourcing** —— 非中介且 `outsourcing_intensity > 0.01`- **Pure Self-Production** —— 其余产出 `firm_classification.png`（左：环形图；右：按年堆叠产出）+ `summary_table.csv/.md`。

In [ ]:
CATS = ["Intermediaries", "Non-intermediaries, outsourcing",        "Non-intermediaries, pure self-produce"]SHORT = dict(zip(CATS, ["Intermediary", "Outsourcing", "Pure Self-Production"]))COLORS = dict(zip(CATS, ["#E07B54", "#4C8BB3", "#5BAD72"]))DEFS = dict(zip(CATS, [">90% output from outsourcing",                       ">1% output from outsourcing (excl. intermediary)",                       "<=1% outsourcing"]))fy = load_firm_year()fy["category"] = np.where(fy["is_intermediary"] == 1, CATS[0],                   np.where(fy["outsourcing_intensity"] > 0.01, CATS[1], CATS[2]))print(f"firm-year: {len(fy):,}   years={sorted(fy['year'].unique())}")# ── 汇总表 ──pri = {c: i for i, c in enumerate(CATS)}firm_dom = (fy[["firm_id", "category"]].assign(p=lambda d: d["category"].map(pri))              .groupby("firm_id")["p"].min().map({i: c for c, i in pri.items()}))uniq = firm_dom.value_counts().reindex(CATS, fill_value=0)agg = (fy.groupby("category")         .agg(firm_year_obs=("firm_id", "count"),              total_output=("firm_total_output", "sum"),              median_output=("firm_total_output", "median"),              mean_os_intensity=("outsourcing_intensity", "mean"))         .reindex(CATS))agg.insert(0, "unique_firms", uniq)agg["total_output_trillion"] = agg["total_output"] / 1e12print("\n── Summary Table (报告 §2.1) ──")disp = agg[["unique_firms", "firm_year_obs", "total_output_trillion", "mean_os_intensity"]].copy()for c in ("unique_firms", "firm_year_obs"): disp[c] = disp[c].map("{:,}".format)disp["total_output_trillion"] = disp["total_output_trillion"].map("{:,.2f}".format)disp["mean_os_intensity"] = disp["mean_os_intensity"].map("{:.4f}".format)print(disp.to_string())print(f"\n企业总数 {int(uniq.sum()):,}（应等于 full_data 的 unique firm 数）")agg.to_csv(FIG / "summary_table.csv")lines = ["# Summary Table — 报告 §2.1", "",         f"来源：`pipeline/figure.ipynb` §2.1，读 `full_data.dta`（{len(fy):,} firm-years）。",         "分类：`is_intermediary == 1`（强度 > 0.9）为 Intermediary；非中介中 `outsourcing_intensity > 0.01` 为 Outsourcing。",         "\"Unique Firms\" 用企业跨年的主导类型（并列偏向更外包的一端）。", "",         "| Firm Type | Definition | Unique Firms | Obs (firm×year) | Total Output (trillion) |",         "|---|---|---|---|---|"]for c in CATS:    lines.append(f"| {SHORT[c]} | {DEFS[c]} | {int(agg.loc[c,'unique_firms']):,} | "                 f"{int(agg.loc[c,'firm_year_obs']):,} | {agg.loc[c,'total_output_trillion']:.2f} |")lines.append(f"| **Total** | | **{int(agg['unique_firms'].sum()):,}** | "             f"**{int(agg['firm_year_obs'].sum()):,}** | **{agg['total_output_trillion'].sum():.2f}** |")(FIG / "summary_table.md").write_text("\n".join(lines), encoding="utf-8")print("  ✓ summary_table.csv + .md")

In [ ]:
plt.rcParams.update({"font.family": "sans-serif", "font.size": 11,                     "axes.spines.top": False, "axes.spines.right": False,                     "figure.facecolor": "white", "axes.facecolor": "white",                     "figure.dpi": 120, "savefig.dpi": 200})cnt = fy["category"].value_counts().reindex(CATS, fill_value=0)d   = fy[fy["year"].isin([2017, 2018])]oby = (d.groupby(["year", "category"])["firm_total_output"].sum()         .unstack(fill_value=0).reindex(columns=CATS, fill_value=0))years = sorted(d["year"].unique())fig, axes = plt.subplots(1, 2, figsize=(14, 6))fig.suptitle("Firm Classification: Intermediary · Outsourcing · Pure Self-Production",             fontsize=14, fontweight="bold", y=1.01)ax = axes[0]w, _, at = ax.pie([int(cnt[c]) for c in CATS], colors=[COLORS[c] for c in CATS],                  autopct="%1.1f%%", startangle=90, pctdistance=0.77,                  wedgeprops=dict(width=0.52, edgecolor="white", linewidth=1.5),                  textprops={"fontsize": 9})for t in at: t.set_fontsize(9); t.set_fontweight("bold"); t.set_color("white")ax.legend(w, [f"{SHORT[c]}\n({cnt[c]:,} firm-years)" for c in CATS],          loc="lower center", bbox_to_anchor=(0.5, -0.18), fontsize=9, frameon=False)ax.set_title("A.  Firm Type Composition", fontweight="bold", pad=8)ax = axes[1]def _fmt(x, _):    for div, suf in ((1e12, "T"), (1e9, "B"), (1e6, "M")):        if x >= div: return f"{x/div:.1f}{suf}"    return f"{x:,.0f}"x, bot = np.arange(len(years)), np.zeros(len(years))for c in CATS:    h = oby.loc[years, c].values    ax.bar(x, h, bottom=bot, color=COLORS[c], edgecolor="white",           linewidth=0.5, width=0.45, label=SHORT[c])    for xi, (hh, bb) in enumerate(zip(h, bot)):        s = hh / oby.loc[years[xi]].sum()        if s > 0.04:            ax.text(xi, bb + hh/2, f"{s*100:.1f}%", ha="center", va="center",                    fontsize=9, color="white", fontweight="bold")    bot += hax.set_xticks(x); ax.set_xticklabels([str(y) for y in years])ax.set_ylabel("Total Firm Output")ax.set_title("B.  Total Output by Year & Type", fontweight="bold", pad=8)ax.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)ax.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt))plt.tight_layout()fig.savefig(FIG / "firm_classification.png", dpi=150, bbox_inches="tight")plt.close(fig)print("✓ firm_classification.png")del fy; gc.collect()

## §2.2　Sales scope vs Production scope产品按外包份额分三类：`pure_self` (OS=0) / `mixed` (0<OS<1) / `pure_os` (OS=1)。$$\text{Sales scope}=n^{self}+n^{mixed}+n^{pure\text{-}OS},\qquad\text{Production scope}=n^{self}+n^{mixed}$$两者之差就是 **scope gap**。2017 / 2018 / pooled 各跑一遍，出 3 张表 + 2 张图。

In [ ]:
CAT_B  = ["pure_self", "mixed", "pure_os"]LAB_B  = dict(zip(CAT_B, ["Pure Self-Production", "Mixed (0 < OS < 1)", "Pure Outsourcing"]))COL_B  = {"Pure Self-Production": "#2ecc71", "Mixed (0 < OS < 1)": "#f39c12",          "Pure Outsourcing": "#e74c3c"}def scope_gap_spec(df_sub, tag, title):    print("\n" + "#"*62)    print(f"#  {tag}   ({title})    {len(df_sub):,} 行")    print("#"*62)    # TABLE 1 产品层面分解    t1 = (df_sub.groupby("category").agg(n_pairs=("firm_id", "size"),                                         total_sales=("total_output", "sum"))                .reindex(CAT_B).reset_index())    t1["pct_pairs"] = t1["n_pairs"] / t1["n_pairs"].sum() * 100    t1["pct_sales"] = t1["total_sales"] / t1["total_sales"].sum() * 100    t1["total_sales_tn"] = t1["total_sales"] / 1e12    t1["label"] = t1["category"].map(LAB_B)    o1 = t1[["label", "n_pairs", "pct_pairs", "total_sales_tn", "pct_sales"]]    o1.columns = ["Category", "N pairs", "% pairs", "Total sales (trillion)", "% sales"]    o1 = pd.concat([o1, pd.DataFrame([{"Category": "Total",        "N pairs": o1["N pairs"].sum(), "% pairs": 100.0,        "Total sales (trillion)": o1["Total sales (trillion)"].sum(),        "% sales": 100.0}])], ignore_index=True)    print(f"\n--- TABLE 1 ({tag}) ---"); print(o1.to_string(index=False, float_format="%.3f"))    o1.to_csv(FIG / f"table1_product_decomposition_{tag}.csv", index=False)    # firm × year 汇总    fc = df_sub.groupby(["firm_id", "year", "category"]).size().unstack(fill_value=0)    fs = df_sub.groupby(["firm_id", "year", "category"])["total_output"].sum().unstack(fill_value=0)    for c in CAT_B:        if c not in fc.columns: fc[c] = 0        if c not in fs.columns: fs[c] = 0    fc = fc[CAT_B]; fc.columns = ["n_pure_self", "n_mixed", "n_pure_os"]    fs = fs[CAT_B]; fs.columns = ["sales_pure_self", "sales_mixed", "sales_pure_os"]    firm = pd.concat([fc, fs], axis=1)    firm["sales_scope"]      = firm[["n_pure_self", "n_mixed", "n_pure_os"]].sum(1)    firm["production_scope"] = firm["n_pure_self"] + firm["n_mixed"]    firm["gap"]              = firm["n_pure_os"]    firm["gap_count_share"]  = firm["gap"] / firm["sales_scope"]    firm["sales_total"]      = firm[["sales_pure_self", "sales_mixed", "sales_pure_os"]].sum(1)    firm["gap_sales_share"]  = np.where(firm["sales_total"] > 0,                                        firm["sales_pure_os"] / firm["sales_total"], np.nan)    # TABLE 2 企业层面分布    desc = lambda s: pd.Series({"mean": s.mean(), "median": s.median(),                                "p75": s.quantile(.75), "p90": s.quantile(.90),                                "p99": s.quantile(.99)})    t2 = pd.DataFrame({        "Sales scope (# products)":      desc(firm["sales_scope"]),        "Production scope (# products)": desc(firm["production_scope"]),        "# Pure Self":                   desc(firm["n_pure_self"]),        "# Mixed":                       desc(firm["n_mixed"]),        "# Pure Outsourcing":            desc(firm["n_pure_os"]),        "Gap (# Pure OS products)":      desc(firm["gap"]),        "Gap / Sales scope (count, %)":  desc(firm["gap_count_share"] * 100),        "Gap / Sales (value, %)":        desc(firm["gap_sales_share"] * 100),    }).T[["mean", "median", "p75", "p90", "p99"]]    print(f"\n--- TABLE 2 ({tag}) ---"); print(t2.to_string(float_format="%.3f"))    t2.to_csv(FIG / f"table2_firm_level_scope_{tag}.csv")    n = len(firm)    pct_gap = (firm["sales_scope"] > firm["production_scope"]).sum() / n * 100    unit = "firm-years" if tag == "pooled" else "firms"    print(f"\nTotal {unit}: {n:,}")    for lab, col in [(">=1 Pure Self", "n_pure_self"), (">=1 Mixed", "n_mixed"),                     (">=1 Pure OS", "n_pure_os")]:        k = (firm[col] >= 1).sum(); print(f"  {lab:<16} {k:,} ({k/n*100:.1f}%)")    print(f"  Scope gap > 0    {int(pct_gap/100*n):,} ({pct_gap:.1f}%)")    # TABLE 3 按规模十分位    firm["firm_output"] = df_sub.groupby(["firm_id", "year"])["firm_total_output"].first()    fd = firm.dropna(subset=["firm_output"]); fd = fd[fd["firm_output"] > 0].copy()    fd["size_decile"] = pd.qcut(fd["firm_output"], 10, labels=False, duplicates="drop") + 1    t3 = fd.groupby("size_decile").agg(        n_units=("sales_scope", "size"),        mean_sales_scope=("sales_scope", "mean"),        mean_prod_scope=("production_scope", "mean"),        mean_gap=("gap", "mean"),        mean_gap_cnt_pct=("gap_count_share", lambda s: s.mean()*100),        mean_gap_val_pct=("gap_sales_share", lambda s: s.mean()*100))    print(f"\n--- TABLE 3 ({tag}) ---"); print(t3.to_string(float_format="%.3f"))    t3.to_csv(FIG / f"table3_by_size_decile_{tag}.csv")    # FIG 2 组合构成    bins = [0.5, 1.5, 2.5, 3.5, 5.5, 10.5, 20.5, 50.5, np.inf]    labs = ["1", "2", "3", "4-5", "6-10", "11-20", "21-50", ">50"]    firm["scope_bin"] = pd.cut(firm["sales_scope"], bins=bins, labels=labs)    compo = firm.groupby("scope_bin", observed=True).agg(        mean_pure_self=("n_pure_self", "mean"), mean_mixed=("n_mixed", "mean"),        mean_pure_os=("n_pure_os", "mean"), n_units=("n_pure_self", "size"))    fig, ax = plt.subplots(figsize=(9, 5.5))    xs, bottom = np.arange(len(compo)), np.zeros(len(compo))    for col, lab in zip(["mean_pure_self", "mean_mixed", "mean_pure_os"], LAB_B.values()):        v = compo[col].values        ax.bar(xs, v, bottom=bottom, label=lab, color=COL_B[lab],               edgecolor="white", linewidth=0.3)        bottom += v    ax.set_xticks(xs); ax.set_xticklabels(compo.index)    ax.set_xlabel("Sales Scope Bin (# products sold)")    ax.set_ylabel(f"Average # products per {'firm-year' if tag=='pooled' else 'firm'}")    ax.set_title(f"Product Composition within Sales Scope Bins — {title}")    ax.legend(loc="upper left")    for i, k in enumerate(compo["n_units"]):        ax.text(xs[i], bottom[i] + bottom.max()*0.01, f"n={k:,}", ha="center", fontsize=8)    plt.tight_layout()    fig.savefig(FIG / f"fig2_composition_by_sales_scope_{tag}.png", bbox_inches="tight")    plt.close(fig); print(f"  ✓ fig2_composition_by_sales_scope_{tag}.png")    # FIG 3 按规模十分位的 gap    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    for ax, col, color, ylab, ttl in [        (axes[0], "mean_gap_cnt_pct", "#3498db", "Mean Gap / Sales Scope (count, %)", "Count-based"),        (axes[1], "mean_gap_val_pct", "#e67e22", "Mean Pure OS Sales / Total Sales (%)", "Sales-based")]:        t3[col].plot(kind="bar", ax=ax, color=color, edgecolor="black")        ax.set_xlabel("Firm Output Decile (1 = smallest)")        ax.set_ylabel(ylab); ax.set_title(f"{ttl} Scope Gap by Firm Size — {title}")        ax.grid(axis="y", alpha=0.3)    plt.tight_layout()    fig.savefig(FIG / f"fig3_gap_by_size_decile_{tag}.png", bbox_inches="tight")    plt.close(fig); print(f"  ✓ fig3_gap_by_size_decile_{tag}.png")    return {"spec": tag, "n_units": n, "n_rows": len(df_sub),            "mean_sales_scope": firm["sales_scope"].mean(),            "mean_prod_scope": firm["production_scope"].mean(),            "mean_gap": firm["gap"].mean(), "pct_with_gap": pct_gap}

In [ ]:
sg = load_scope_gap()head = [scope_gap_spec(sg[sg["year"] == 2017].copy(), "2017", "Year 2017"),        scope_gap_spec(sg[sg["year"] == 2018].copy(), "2018", "Year 2018"),        scope_gap_spec(sg.copy(), "pooled", "Pooled 2017 + 2018")]print("\n" + "="*62); print("CROSS-SPEC COMPARISON"); print("="*62)comp = pd.DataFrame(head).set_index("spec")print(comp.to_string(float_format="%.3f"))comp.to_csv(FIG / "table0_cross_year_comparison.csv")del sg; gc.collect()

## §3.1a　销售份额 vs 相似度副产品的销售份额，三种分母：| | 分母 ||---|---|| **A** | 企业总销售额 || **B** | 企业副产品销售额合计 || **C** | 核心产品销售额 |按相似度分 25 个等宽 bin，画均值 + 95% CI。**论文用的是 combined 那两张。**

In [ ]:
NBINS = 25plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12,                     "axes.titlesize": 13, "axes.labelsize": 12})sec = load_secondary()sec["sec_total"] = sec.groupby(["firm_id", "year"])["total_output"].transform("sum")# 核心产品销售额（分母 C）—— 复用 load_core，不再单独扫全表sec = sec.merge(load_core()[["firm_id", "year", "main_sales"]],                on=["firm_id", "year"], how="left")sec["share_A"] = sec["total_output"] / sec["firm_total_output"]sec["share_B"] = sec["total_output"] / sec["sec_total"]sec["share_C"] = sec["total_output"] / sec["main_sales"]print(f"副产品 {len(sec):,} 行")sim_bi = load_sim_bi()ss = sec.merge(sim_bi, on=["main_pid", "product_id"], how="inner")print(f"并上相似度后 {len(ss):,} 行")del sec; gc.collect()

In [ ]:
SHARES = {"A": ("share_A", "Product Share of Firm Total Sales",   "steelblue"),          "B": ("share_B", "Product Share of Firm Secondary Sales","darkorange"),          "C": ("share_C", "Product Sales / Core Product Sales",   "forestgreen")}SIMS   = {"input":  ("input_similarity",  "Input Similarity"),          "output": ("output_similarity", "Demand Complementarity")}def binstats(sub, simcol, sharecol):    s = sub[[simcol, sharecol]].dropna().copy()    s["bin"] = pd.cut(s[simcol], bins=NBINS, include_lowest=True)    st = s.groupby("bin", observed=True)[sharecol].agg(["mean", "sem"]).reset_index()    st["mid"] = st["bin"].apply(lambda x: x.mid)    return st.dropna(subset=["mean"])# 单张（6 张，备用）for sk, (simcol, simlab) in SIMS.items():    for shk, (shcol, shlab, color) in SHARES.items():        st = binstats(ss, simcol, shcol)        fig, ax = plt.subplots()        w = (st["mid"].iloc[1] - st["mid"].iloc[0]) * 0.85        ax.bar(st["mid"], st["mean"], width=w, color=color, alpha=.75)        ax.errorbar(st["mid"], st["mean"], yerr=1.96*st["sem"],                    fmt="none", color="black", capsize=3, linewidth=.8)        ax.set_xlabel(simlab); ax.set_ylabel(shlab)        ax.set_title(f"Figure 1{shk}: Mean {shlab}\nby {simlab} Bin")        fig.tight_layout()        fig.savefig(FIG / f"fig1{shk}_{shcol}_vs_{sk}_similarity.png", dpi=150)        plt.close(fig)    print(f"  ✓ fig1A/B/C — {simlab}")# combined（论文用）for sk, (simcol, simlab) in SIMS.items():    fig, axes = plt.subplots(1, 3, figsize=(18, 5))    fig.suptitle(f"Mean Sales Share by {simlab} Bin (Three Denominators)", fontsize=14)    for ax, (shk, (shcol, shlab, color)) in zip(axes, SHARES.items()):        st = binstats(ss, simcol, shcol)        w = (st["mid"].iloc[1] - st["mid"].iloc[0]) * 0.85        ax.bar(st["mid"], st["mean"], width=w, color=color, alpha=.75)        ax.errorbar(st["mid"], st["mean"], yerr=1.96*st["sem"],                    fmt="none", color="black", capsize=2, linewidth=.7)        ax.set_xlabel(simlab); ax.set_ylabel(shlab); ax.set_title(f"Denominator {shk}")    fig.tight_layout()    fig.savefig(FIG / f"fig1_combined_shares_vs_{sk}_similarity.png", dpi=150)    plt.close(fig)    print(f"  ✓ fig1_combined_shares_vs_{sk}_similarity.png")

## §3.1b　选择提升 Lift$$\text{Lift(bin)}=\frac{P(\text{chosen}\mid \text{bin})}{P(\text{chosen})}$$分母是全样本无条件选择概率。对每个产品对 $(m,c)$，`n_exposure` = 以 $m$ 为核心产品的 firm-year 数，`n_chosen` = 其中同时生产 $c$ 的数量。**这张图（combined 版）同时进了论文和 deck。**

In [ ]:
core = load_core()sec2 = load_secondary()[["firm_id", "year", "product_id", "main_pid"]]expo = core.groupby("main_pid").size().reset_index(name="n_exposure")chos = sec2.groupby(["main_pid", "product_id"]).size().reset_index(name="n_chosen")print(f"核心产品数 {expo['main_pid'].nunique():,}   被选产品对 {len(chos):,}")lift = load_sim_bi().merge(expo, on="main_pid", how="left").dropna(subset=["n_exposure"])lift["n_exposure"] = lift["n_exposure"].astype(int)lift = lift.merge(chos, on=["main_pid", "product_id"], how="left")lift["n_chosen"] = lift["n_chosen"].fillna(0).astype(int)g_rate = lift["n_chosen"].sum() / lift["n_exposure"].sum()print(f"全局 P(chosen) = {lift['n_chosen'].sum():,} / {lift['n_exposure'].sum():,} = {g_rate:.6f}")del core, sec2, expo, chos; gc.collect()

In [ ]:
aggs = {}for simcol, simlab, c1 in [("input_similarity",  "Input Similarity",      "#2166ac"),                           ("output_similarity", "Demand Complementarity","#d6604d")]:    s = lift[[simcol, "n_chosen", "n_exposure"]].dropna().copy()    s["bin"] = pd.cut(s[simcol], bins=NBINS, include_lowest=True)    a = s.groupby("bin", observed=True).agg(n_chosen=("n_chosen", "sum"),                                            n_exposure=("n_exposure", "sum"),                                            n_pairs=("n_chosen", "count")).reset_index()    a["mid"]      = a["bin"].apply(lambda x: x.mid)    a["p_chosen"] = a["n_chosen"] / a["n_exposure"]    a["lift"]     = a["p_chosen"] / g_rate    a["ci"] = 1.96 * np.sqrt(a["p_chosen"]*(1-a["p_chosen"])/a["n_exposure"]) / g_rate    print(f"\n── {simlab} ──")    print(a[["mid", "n_pairs", "n_chosen", "p_chosen", "lift"]].to_string(index=False))    fig, ax = plt.subplots(figsize=(10, 5))    w = (a["mid"].iloc[1] - a["mid"].iloc[0]) * 0.85    ax.bar(a["mid"], a["lift"], width=w, color=c1, alpha=.8, label="Lift")    ax.errorbar(a["mid"], a["lift"], yerr=a["ci"], fmt="none",                color="black", capsize=3, linewidth=.8)    ax.axhline(1.0, color="gray", ls="--", lw=1.2, label="Baseline (lift = 1)")    ax.set_xlabel(simlab); ax.set_ylabel("Lift = P(chosen | bin) / P(chosen)")    ax.set_title(f"Figure 2: Selection Lift by {simlab} Bin")    ax.legend(); fig.tight_layout()    fig.savefig(FIG / f"fig2_lift_{simcol}.png", dpi=150)    plt.close(fig); print(f"  ✓ fig2_lift_{simcol}.png")    aggs[simcol] = (a, c1, simlab)fig, axes = plt.subplots(1, 2, figsize=(16, 5))fig.suptitle("Figure 2: Selection Lift vs. Similarity (Conditional Probability / Baseline)",             fontsize=13)for ax, key in zip(axes, ["input_similarity", "output_similarity"]):    a, color, simlab = aggs[key]    w = (a["mid"].iloc[1] - a["mid"].iloc[0]) * 0.85    ax.bar(a["mid"], a["lift"], width=w, color=color, alpha=.8)    ax.errorbar(a["mid"], a["lift"], yerr=a["ci"], fmt="none",                color="black", capsize=2, linewidth=.7)    ax.axhline(1.0, color="gray", ls="--", lw=1.2)    ax.set_xlabel(simlab); ax.set_ylabel("Lift"); ax.set_title(simlab)fig.tight_layout()fig.savefig(FIG / "fig2_lift_combined.png", dpi=150)plt.close(fig); print("  ✓ fig2_lift_combined.png")del lift; gc.collect()

## §3.2　相似度 vs 销量排名副产品按企业内销量排序（rank 1 = 最大的副产品），看相似度随排名怎么变。**结论**：两个维度都随排名**单调下降**——企业最大的副产品，系统性地最像核心产品。按副产品个数分组是为了排除企业异质性带来的构成效应。`sim_vs_rank_all.png` 同时进了论文和 deck。

In [ ]:
sec3 = load_secondary()sec3["n_sec"] = sec3.groupby(["firm_id", "year"])["product_id"].transform("count")sec3["sales_rank"] = (sec3.groupby(["firm_id", "year"])["total_output"]                          .rank(method="first", ascending=False).astype(int))sec3 = sec3.merge(load_sim_bi(), on=["main_pid", "product_id"], how="left")sec3 = sec3.dropna(subset=["input_similarity"])print(f"并上相似度后 {len(sec3):,} 行")GROUPS = {    "all":      ("Full Sample",              sec3),    "n_le5":    ("1-5 Secondary Products",   sec3[sec3["n_sec"] <= 5]),    "n_6to10":  ("6-10 Secondary Products",  sec3[(sec3["n_sec"] >= 6) & (sec3["n_sec"] <= 10)]),    "n_11to15": ("11-15 Secondary Products", sec3[(sec3["n_sec"] >= 11) & (sec3["n_sec"] <= 15)]),}for key, (label, sub) in GROUPS.items():    if len(sub) == 0:        print(f"  跳过 {key}（无数据）"); continue    mr = min(int(sub["sales_rank"].quantile(0.95)), 30)    pd_ = sub[sub["sales_rank"] <= mr]    st = {}    for col in ("input_similarity", "output_similarity"):        a = pd_.groupby("sales_rank")[col].agg(["mean", "std", "count"]).reset_index()        a["se"] = a["std"] / np.sqrt(a["count"])        st[col] = a    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))    for ax, col, color, lab in [        (ax1, "input_similarity",  "steelblue",  "Input Similarity"),        (ax2, "output_similarity", "darkorange", "Demand Complementarity")]:        a = st[col]        ax.bar(a["sales_rank"], a["mean"], width=.7, color=color,               edgecolor="white", alpha=.85)        ax.errorbar(a["sales_rank"], a["mean"], yerr=1.96*a["se"],                    fmt="none", color="black", capsize=2, linewidth=.8)        ax.set_xlabel("Sales Rank (1 = Largest Secondary Product)")        ax.set_ylabel(f"Mean {lab} to Core Product")        ax.set_title(f"{lab} — {label}")        ax.set_xlim(0, mr + 1)    fig.suptitle(f"Product Similarity vs. Sales Rank of Secondary Products\n({label})",                 fontsize=15, y=1.02)    fig.tight_layout()    fig.savefig(FIG / f"sim_vs_rank_{key}.png", dpi=150, bbox_inches="tight")    plt.close(fig)    print(f"  ✓ sim_vs_rank_{key}.png   (N={len(pd_):,}, max_rank={mr})")    # 双序列并排版    fig, ax = plt.subplots(figsize=(10, 6))    bw = 0.35; x = st["input_similarity"]["sales_rank"].values    for off, col, color, lab in [(-bw/2, "input_similarity",  "steelblue",  "Input Similarity"),                                 (+bw/2, "output_similarity", "darkorange", "Demand Complementarity")]:        a = st[col]        ax.bar(x + off, a["mean"].values, width=bw, color=color,               edgecolor="white", alpha=.85, label=lab)        ax.errorbar(x + off, a["mean"].values, yerr=1.96*a["se"].values,                    fmt="none", color="black", capsize=1.5, linewidth=.6)    ax.set_xlabel("Sales Rank (1 = Largest Secondary Product)")    ax.set_ylabel("Mean Similarity to Core Product")    ax.set_title(f"Similarity vs. Sales Rank — {label}")    ax.legend(); ax.set_xlim(0, mr + 1)    fig.tight_layout()    fig.savefig(FIG / f"sim_vs_rank_{key}_combined.png", dpi=150)    plt.close(fig)    print(f"  ✓ sim_vs_rank_{key}_combined.png")del sec3; gc.collect()

## §IO　IO 向量与夹角图（★ 重写）原脚本 `Data/seminar_viz.py` 已从机器上删除，全盘搜不到、无 git 历史，这里重写。### 四处修正**① 系数列**　去向向量必须用 `coefficient_cal`。校准 = `coefficient_cal = coefficient × 只依赖于行的标量`（行内比值相对标准差 5.15e-16）。S 用行向量，逐行同乘不改余弦，**对校准免疫**；C 用列向量，**完全依赖校准**。raw 表行和中位数 1.109 但最大 3035.8（极差 3.27 万倍），且 `corr(log 产出规模, log 行和) = −0.59`——raw 是**反着加权**的，最小的部门主导每一列。**② 例子**　原图用乘用车 / 电动汽车 / 碾磨脱壳谷物。老师不喜欢，数据也不支持：乘用车—电动汽车 S=0.8197（第 7/2777）、**C=0.2381（也是第 7/2777，p99.59）**——是高 S **高** C。**③ 图的设计**　原来是「A 锚定 / B 投入像 / C 投入不像」，两个面板用同一组产品。问题是 C 在需求维度上并不低（毛巾—碾磨 C=0.047 反而比毛巾—针织袜的 0.037 高），右面板两条线挤在一起。改成 **2×2 对照**——B 高 S 低 C、D 低 S 高 C，两个面板的排序**正好相反**，才真正说明两个维度正交：| 与锚定产品 A = 毛巾 | S | C ||---|---|---|| **B = 针织袜** | **0.991** | 0.037 || **D = 天然皮革服装** | 0.015 | **0.208** |同样的纺织投入、不同的买家 ⟷ 完全不同的生产、相同的零售渠道。**④ 标签**　`Output Similarity` → `Demand Complementarity`；产品名走 `TRANS` 字典出英文（论文是英文的）。漏译的会打印出来，补进字典即可；补不全就回落中文 + 中文字体。

In [ ]:
if not AVAIL["io_table_calibrated.dta"]:    print("⚠️  缺 io_table_calibrated.dta，本节全部跳过。")    print("    从本地 C:\\Users\\HKUBS\\Documents\\aproject\\Data\\ 拷 1.8 GB 过来。")    print("    不能用 io_table_lite.dta 顶替——它只有 coefficient，没有 coefficient_cal。")else:    names = (pd.read_stata(BIANMA).rename(columns={"货物和劳务名称": "name"})               .set_index("product_id")["name"])    pid  = sorted(names.index.astype(str))    idx  = {p: i for i, p in enumerate(pid)}    N    = len(pid)    print(f"读 io_table_calibrated.dta（{N}×{N}）…")    A_cal = np.zeros((N, N))    for i, ch in enumerate(pd.read_stata(IO_CAL, columns=["product_id", "product_id_input",                                                          "coefficient_cal"],                                         chunksize=2_000_000), 1):        ch = ch[ch.product_id.isin(idx) & ch.product_id_input.isin(idx)]        A_cal[ch.product_id.map(idx).values,              ch.product_id_input.map(idx).values] = ch.coefficient_cal.values    print(f"  非零 {int((A_cal>0).sum()):,}   稠密度 {(A_cal>0).mean():.1%}")    def S(a, b):    # input similarity = 行余弦（投入结构）        x, y = A_cal[idx[a]], A_cal[idx[b]]        return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))    def C(a, b):    # demand complementarity = 列余弦（下游用途）        x, y = A_cal[:, idx[a]], A_cal[:, idx[b]]        return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))    name2id = {v: k for k, v in names.items()}

In [ ]:
# ── 换锚定产品时用这一格找搭档 ──ANCHOR_NAME = "毛巾"if AVAIL["io_table_calibrated.dta"]:    def _cosmat(M):        n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1        return (M/n) @ (M/n).T    CS, CC = _cosmat(A_cal), _cosmat(A_cal.T)    np.fill_diagonal(CS, np.nan); np.fill_diagonal(CC, np.nan)    a = idx[name2id[ANCHOR_NAME]]    t = pd.DataFrame({"名称": [names.get(p, "?") for p in pid],                      "S": CS[a], "C": CC[a]}).dropna()    print(f"锚定 = {ANCHOR_NAME}")    print("全表参考：S p50=0.050 p99=0.516  |  C p50=0.0018 p90=0.033 p99=0.154\n")    print("── B 候选：高 S 低 C（投入像、需求不像）──")    print(t[(t.S > 0.9) & (t.C < 0.05)].nlargest(6, "S")           .to_string(index=False, float_format="%.4f"))    print("\n── D 候选：低 S 高 C（投入不像、需求像）──")    print(t[t.S < 0.15].nlargest(8, "C").to_string(index=False, float_format="%.4f"))    del CS, CC; gc.collect()

In [ ]:
# ════════ 选定的三个产品 ════════#   A = 锚定    B = 高 S 低 C    D = 低 S 高 CTRIPLE = {"A": "毛巾", "B": "针织袜", "D": "天然皮革服装"}# 产品名 → 英文（论文是英文的）。漏的会打印出来，补进这里即可。TRANS = {    "毛巾": "Towels", "针织袜": "Knitted socks", "天然皮革服装": "Leather apparel",    "棉纱线": "Cotton yarn", "棉机织物": "Woven cotton fabric", "棉": "Cotton",    "化纤机织物": "Woven synthetic fabric", "针织物": "Knitted fabric",    "化学纤维": "Chemical fibre", "染料": "Dyes", "印染加工": "Dyeing services",    "电力": "Electricity", "蒸汽": "Steam", "柴油": "Diesel", "汽油": "Gasoline",    "塑料薄膜": "Plastic film", "纸制品": "Paper products", "瓦楞纸箱": "Corrugated boxes",    "天然皮革": "Natural leather", "毛皮": "Fur", "鞋": "Footwear",    "道路货物运输服务": "Road freight", "仓储服务": "Warehousing",    "其他现代服务": "Other modern services", "咨询服务": "Consulting",}EN = lambda z: TRANS.get(z, z)if AVAIL["io_table_calibrated.dta"]:    missing = [k for k, v in TRIPLE.items() if v not in name2id]    if missing:        print(f"❌ 找不到产品名：{[TRIPLE[k] for k in missing]}")        print("   用上一格挑，或搜：names[names.str.contains('毛巾')]")    else:        P = {k: name2id[v] for k, v in TRIPLE.items()}        print(f"{'':5}{'产品':<16}{'英文':<20}{'代码'}")        for k in "ABD":            print(f"  {k}: {TRIPLE[k]:<16}{EN(TRIPLE[k]):<20}{P[k]}")        print(f"\n{'':16}{'S (input)':>12}{'C (demand)':>14}")        for k in "BD":            print(f"  A—{k}  {TRIPLE[k]:<10}{S(P['A'],P[k]):>12.4f}{C(P['A'],P[k]):>14.4f}")        print(f"\n  ✓ 两个维度排序相反 → 面板会翻转"              if (S(P['A'],P['B']) > S(P['A'],P['D'])) and (C(P['A'],P['B']) < C(P['A'],P['D']))              else "\n  ⚠️ 两个维度排序相同，翻转效果不成立，换一组")

In [ ]:
# ════════ fig1_input_vectors.png —— 三个产品的投入结构 ════════if AVAIL["io_table_calibrated.dta"] and not missing:    # 中文回落字体（漏译时仍可读）    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]    plt.rcParams["axes.unicode_minus"] = False    TOPK, untranslated = 8, set()    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))    fig.suptitle("Input coefficients of three products "                 "(largest entries of each product's input vector)",                 fontsize=14, y=1.03)    for ax, k, col in zip(axes, "ABD", ["#333333", "#2166ac", "#d6604d"]):        v = pd.Series(A_cal[idx[P[k]]], index=pid)        v = v[v > 0].nlargest(TOPK)[::-1]        zh = [str(names.get(p, p)) for p in v.index]        untranslated |= {z for z in zh if z not in TRANS}        ax.barh(range(len(v)), v.values, color=col, edgecolor="white", alpha=.85)        ax.set_yticks(range(len(v)))        ax.set_yticklabels([EN(z)[:26] for z in zh], fontsize=9)        ax.set_xlabel("Input coefficient (share of output value)")        ax.set_title(f"{k}.  {EN(TRIPLE[k])}", fontweight="bold", fontsize=12)        ax.spines[["top", "right"]].set_visible(False)        ax.grid(axis="x", alpha=.25)    fig.tight_layout()    fig.savefig(FIG / "fig1_input_vectors.png", dpi=150, bbox_inches="tight")    plt.close(fig)    print("  ✓ fig1_input_vectors.png")    if untranslated:        print(f"\n  ⚠️ 这 {len(untranslated)} 个投入品名没翻译（图里显示中文），补进 TRANS 即可：")        for z in sorted(untranslated): print(f'      "{z}": "",')

In [ ]:
# ════════ fig3_vector_angles.png —— 两个维度作为向量夹角 ════════if AVAIL["io_table_calibrated.dta"] and not missing:    COL = {"A": "#333333", "B": "#2166ac", "D": "#d6604d"}    def draw(ax, sims, title, blurb):        """sims: {'B': cos, 'D': cos}。A 竖直向上，其余按夹角顺时针摊开。"""        # A 与自己相似度 = 1 → 夹角 0 → 竖直向上；其余按夹角顺时针摊开        items = [("A", 1.0)] + sorted(sims.items(), key=lambda kv: -kv[1])        placed = []        for key, cos in items:            ang = np.arccos(np.clip(cos, -1, 1))          # 与 A 的夹角            th  = np.pi/2 - ang            v   = np.array([np.cos(th), np.sin(th)])            ax.annotate("", xy=v, xytext=(0, 0),                        arrowprops=dict(arrowstyle="-|>", color=COL[key],                                        lw=2.6 if key == "A" else 2.2))            # 标签半径：与已放置的标签角度太近就往外推，避免叠字            r = 1.06            for pth, pr in placed:                if abs(th - pth) < 0.28: r = max(r, pr + 0.21)            placed.append((th, r))            ax.text(r*np.cos(th), r*np.sin(th),                    f"{key}   {EN(TRIPLE[key])}", fontsize=11.5, color=COL[key],                    ha="left", va="center", fontweight="bold")        # 夹角弧，两条用不同半径        for n, (key, cos) in enumerate(sorted(sims.items(), key=lambda kv: -kv[1])):            ang = np.arccos(np.clip(cos, -1, 1))            rad = 0.30 + 0.16*n            arc = np.linspace(np.pi/2 - ang, np.pi/2, 80)            ax.plot(rad*np.cos(arc), rad*np.sin(arc), color=COL[key], lw=1.7)            mid = arc.mean()            ax.text((rad+0.09)*np.cos(mid), (rad+0.09)*np.sin(mid),                    f"{np.degrees(ang):.0f}°", color=COL[key],                    fontsize=11, fontweight="bold", ha="center", va="center")        ax.set_xlim(-0.10, 1.95); ax.set_ylim(-0.12, 1.30)        ax.set_aspect("equal"); ax.axis("off")        sub = "      ".join(f"A–{k} = {v:.3f}" for k, v in sims.items())        ax.set_title(f"{title}\n{sub}", fontsize=13, fontweight="bold", pad=14)        ax.text(0.5, -0.06, blurb, transform=ax.transAxes, ha="center", va="top",                fontsize=9.5, bbox=dict(boxstyle="round,pad=0.55",                                        facecolor="#F4F4F4", edgecolor="#CCCCCC"))    fig, axes = plt.subplots(1, 2, figsize=(15, 7))    draw(axes[0], {"B": S(P["A"], P["B"]), "D": S(P["A"], P["D"])},         "Input Similarity  (S)",         "Cosine between the products' INPUT vectors.\n"         "High = same materials, processes, suppliers\n=> cheaper to make in-house.")    draw(axes[1], {"B": C(P["A"], P["B"]), "D": C(P["A"], P["D"])},         "Demand Complementarity  (C)",         "Cosine between the products' DOWNSTREAM-USE vectors.\n"         "High = bought by the same industries\n=> the existing customer base carries the new product.")    fig.suptitle("The two similarity measures as angles between product vectors",                 fontsize=15, y=1.00)    fig.tight_layout(rect=[0, 0.07, 1, 0.96])    fig.savefig(FIG / "fig3_vector_angles.png", dpi=150, bbox_inches="tight")    plt.close(fig)    print("  ✓ fig3_vector_angles.png")    print(f"    左panel: A–B {S(P['A'],P['B']):.3f} > A–D {S(P['A'],P['D']):.3f}")    print(f"    右panel: A–B {C(P['A'],P['B']):.3f} < A–D {C(P['A'],P['D']):.3f}   ← 翻转")

## 完成所有图落在 `Empirical1/results/figures/`（进 git，本地 pull 即可看）。中间件在 `Empirical1_data/_figure_cache/`（不进 git），删掉即可强制重算。### 与论文的对应| 论文 | 本 notebook 产出 ||---|---|| Fig 1 (input vectors) | `fig1_input_vectors.png` ★重写 || Fig (firm classification) | `firm_classification.png` || Fig (scope) | `fig2_composition_by_sales_scope_pooled.png` || Fig (scope gap by size) | `fig3_gap_by_size_decile_pooled.png` || Fig (sales share) | `fig1_combined_shares_vs_{input,output}_similarity.png` || Fig (lift) | `fig2_lift_combined.png` || Fig (sim vs rank) | `sim_vs_rank_all.png` || Fig (vector angles) | `fig3_vector_angles.png` ★重写 |